In [29]:
import os, time
import requests
import pandas as pd

START, END = "2025-01-01", "2025-01-31"   # 1 month = 31 days

# name -> (lat, lon, EIA respondent code)
ZONES = {
    "Maryland_PJM":     (39.29, -76.61, "PJM"),
    "NewYork_NYISO":    (40.71, -74.00, "NYIS"),
    "California_CAISO": (34.05, -118.24, "CISO"),
}

EIA_KEY = os.environ.get("EIA_API_KEY")


class ClimateEnergyPipeline:
    def __init__(self):
        os.makedirs("data/raw", exist_ok=True)

    def save(self, name, kind, df):
        """Save raw data as CSV only."""
        df["region"] = name
        df.to_csv(f"data/raw/{name}_{kind}.csv", index=False)
        print(f"  Saved {name}_{kind}.csv ({len(df)} rows)")

    def fetch_weather(self, name, lat, lon):
        print(f"Fetching weather for {name}...")
        params = {
            "latitude": lat, "longitude": lon, "timezone": "UTC",
            "start_date": START, "end_date": END,
            "hourly": "temperature_2m,direct_normal_irradiance,wind_speed_100m",
        }
        r = requests.get("https://archive-api.open-meteo.com/v1/archive",
                         params=params, timeout=30)
        r.raise_for_status()
        self.save(name, "weather", pd.DataFrame(r.json()["hourly"]))

    def fetch_energy(self, name, code):
        print(f"Fetching energy for {name}...")
        url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"
        rows, offset = [], 0
        while True:  # EIA returns max 5000 rows per request
            params = {
                "api_key": EIA_KEY,
                "frequency": "hourly",
                "data[]": "value",
                "facets[respondent][]": code,
                "facets[type][]": "D",  # D = demand
                "start": f"{START}T00",
                "end": f"{END}T23",
                "sort[0][column]": "period",
                "sort[0][direction]": "asc",
                "offset": offset,
                "length": 5000,
            }
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            batch = r.json()["response"]["data"]
            rows.extend(batch)
            if len(batch) < 5000:
                break
            offset += 5000
        self.save(name, "energy", pd.DataFrame(rows))

    def run(self):
        for name, (lat, lon, code) in ZONES.items():
            try:
                self.fetch_weather(name, lat, lon)
                if EIA_KEY:
                    self.fetch_energy(name, code)
                else:
                    print("  EIA_API_KEY not set, skipping energy.")
            except Exception as e:
                print(f"  Failed for {name}: {e}")
            time.sleep(1)

In [30]:
ClimateEnergyPipeline().run()
print("\n--- PIPELINE RUN COMPLETED ---")

Fetching weather for Maryland_PJM...
  Saved Maryland_PJM_weather.csv (744 rows)
Fetching energy for Maryland_PJM...
  Saved Maryland_PJM_energy.csv (744 rows)
Fetching weather for NewYork_NYISO...
  Saved NewYork_NYISO_weather.csv (744 rows)
Fetching energy for NewYork_NYISO...
  Saved NewYork_NYISO_energy.csv (744 rows)
Fetching weather for California_CAISO...
  Saved California_CAISO_weather.csv (744 rows)
Fetching energy for California_CAISO...
  Saved California_CAISO_energy.csv (744 rows)

--- PIPELINE RUN COMPLETED ---


In [24]:
import pandas as pd
print(pd.read_csv("data/raw/Maryland_PJM_energy.csv").head())
print(pd.read_csv("data/raw/Maryland_PJM_weather.csv").head())

          period respondent           respondent-name type type-name  value  \
0  2025-01-01T00        PJM  PJM Interconnection, LLC    D    Demand  94357   
1  2025-01-01T01        PJM  PJM Interconnection, LLC    D    Demand  91598   
2  2025-01-01T02        PJM  PJM Interconnection, LLC    D    Demand  88981   
3  2025-01-01T03        PJM  PJM Interconnection, LLC    D    Demand  86432   
4  2025-01-01T04        PJM  PJM Interconnection, LLC    D    Demand  84245   

     value-units        region  
0  megawatthours  Maryland_PJM  
1  megawatthours  Maryland_PJM  
2  megawatthours  Maryland_PJM  
3  megawatthours  Maryland_PJM  
4  megawatthours  Maryland_PJM  
               time  temperature_2m  direct_normal_irradiance  \
0  2025-01-01T00:00             9.3                       0.0   
1  2025-01-01T01:00             8.9                       0.0   
2  2025-01-01T02:00             9.0                       0.0   
3  2025-01-01T03:00             8.9                       0.0   
4 

In [21]:
import os
print(os.path.abspath("data/raw"))

c:\Users\alima\AppData\Local\Programs\Microsoft VS Code\data\raw


In [31]:
os.startfile(os.path.abspath("data/raw"))